In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to POS / Sales (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'POS / Sales')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='POS / Sales']")))

    # Add the first available medicine to the cart (Add button, verified in CatalogTable.jsx)
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr[contains(@class, 'pos-row')]")))
    add_buttons = [b for b in driver.find_elements(By.XPATH, "//tr[contains(@class, 'pos-row')]//button[contains(., 'Add')]") if b.is_displayed() and b.is_enabled()]
    assert add_buttons, "No addable medicine found in the catalog."
    add_buttons[0].click()
    time.sleep(2)

    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Current Sale" in body and "Cart is empty" not in body, "Cart has no item."

    # Select Split payment (method button, verified in PaymentCard.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Split')]"))).click()
    cash_input = wait.until(EC.visibility_of_element_located((By.ID, "pos-split-cash")))
    dig_input = driver.find_element(By.ID, "pos-split-digital")

    # Read the real total from the Complete Sale button text ("Complete Sale \u00b7 ৳X")
    complete_btn = driver.find_element(By.XPATH, "//button[contains(., 'Complete Sale') and not(contains(., 'Reviewed'))]")
    total = int(re.sub(r"[^\d]", "", complete_btn.text))
    print("Sale total:", total)
    cash_amt = total // 2
    dig_amt = total - cash_amt
    cash_input.clear()
    cash_input.send_keys(str(cash_amt))
    dig_input.clear()
    dig_input.send_keys(str(dig_amt))
    time.sleep(2)

    # The app shows "amounts must match the total" only when split != total
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "amounts must match" not in body, "Split amounts do not match the sale total."

    # Complete the sale
    driver.find_element(By.XPATH, "//button[contains(., 'Complete Sale') and not(contains(., 'Reviewed'))]").click()
    time.sleep(3)
    approve = [b for b in driver.find_elements(By.XPATH, "//button[contains(., 'Reviewed')]") if b.is_displayed()]
    if approve:
        print("Approval modal appeared, confirming...")
        approve[0].click()
        time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Sale Completed']")))

    print("Current URL:", driver.current_url)
    print("PASS: Split Payment")
except Exception as e:
    print("FAIL: Split Payment")
    print("Error:", e)
    driver.save_screenshot("17_split_payment_FAIL.png")

In [ ]:
driver.quit()